# CFM — les données et les features en un seul notebook

| Partie | Question | Durée |
|---|---|---|
| 1 · Anatomie | Une vraie fenêtre : carnet, événements, parcours d'ordres, tokens | < 2 min |
| 2 · Signatures | Ce qui distingue les titres : ticks, lots, venues, position, profondeur, empreinte de tokens | < 3 min |
| 3 · Dérive | Ce qui change entre train et test ; hypothèse « prix plus élevés » ; AUC adversariale | ~5 min |
| 4 · Ordres | Comment vivent les ordres, titre par titre | < 2 min |

Pas de réseau de neurones ici (torch n'est pas importé). Le test n'a pas de labels : pour le découper par titre, on utilise le titre **prédit** par la meilleure soumission (V3 B), si `teacher/v3_B_probs.npz` est présent.

In [ ]:
import os, sys, json, time
from pathlib import Path
if sys.platform == 'darwin':
    os.environ.setdefault('OMP_NUM_THREADS', '1')
CODE_DATASET = None   # ex. 'ton-username/cfm-signature-lab' ; None : code local ou Dataset attaché

def find_repo():
    if CODE_DATASET:
        import kagglehub
        path = Path(kagglehub.dataset_download(CODE_DATASET, force_download=True))
        return next(p.parent.parent for p in path.glob('**/cfm/__init__.py'))
    for p in [Path('.'), Path('..'), Path('../..'), Path('/kaggle/working/cfm-signature-lab')]:
        if (p / 'cfm' / '__init__.py').exists():
            return p.resolve()
    for p in Path('/kaggle/input').glob('**/cfm/__init__.py'):
        return p.parent.parent
    raise FileNotFoundError('Code introuvable : renseigner CODE_DATASET ou attacher le Dataset du code')

REPO = find_repo(); sys.path.insert(0, str(REPO))
DEMO = False          # True : données synthétiques sur CPU (vérifie que tout tourne ; scores sans valeur)
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from cfm.demo import core, viz          # pas de torch ici (arbres / exploration)
from cfm import io, screen, plots as cplots
from cfm.features.common import Raw
from cfm.registry import context
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 30)
viz.setup('figures/features')
lab = core.open_lab(DEMO, torch_device=False)
LABELS = [str(c) for c in lab.classes]

## Partie 1 · Anatomie — Anatomie d'une fenêtre

**Question.** À quoi ressemblent concrètement 100 événements ?
- Comment un ordre vit-il dans une fenêtre ?
- Qu'est-ce qui distingue, à l'œil, un titre « à gros tick » d'un titre « à petit tick » ?

Durée : < 2 min.

In [ ]:
raw = io.load(lab.dir, 'train')
X, names = lab.table(['ticks'])
s1 = X[:, names.index('ticks:spread_1')]
cls = pd.Series(s1).groupby(lab.y).mean()
big, small = int(cls.idxmax()), int(cls.idxmin())
print(f'titre au plus gros tick : {LABELS[big]} (spread à 1 tick {cls[big]:.0%} du temps) | au plus petit : {LABELS[small]} ({cls[small]:.0%})')
pick = {f'gros tick (titre {LABELS[big]})': np.flatnonzero(lab.y == big)[0], f'petit tick (titre {LABELS[small]})': np.flatnonzero(lab.y == small)[0]}
for k, i in pick.items():
    viz.window_anatomy({key: np.asarray(raw[key][i]) for key in ['num', 'cat', 'oid']}, io.meta(lab.dir)['tick'],
                       f'Une fenêtre réelle — {k}', 'd1_window_' + ('big' if 'gros' in k else 'small'))

In [ ]:
seq = lab.sequences('new')
i = pick[list(pick)[0]]
viz.token_strip(seq['tok'][i], [n.replace('tok_', '') for n in core.NEW_TOKENS], title='Les 40 premiers événements de la même fenêtre, en tokens', name='d1_tokens')
r = Raw({k: np.asarray(raw[k][i:i + 1]) for k in ['num', 'cat', 'oid']}, context(lab.dir, lab.cfg['lot']))
pd.DataFrame({'action': r.action[0], 'côté': r.side[0], 'trade': r.is_trade[0], 'spread (ticks)': r.spread_t[0],
              'Δmid (½ ticks)': r.dmid_h[0], 'distance (ticks)': r.own_dist[0], 'flux (lots)': r.flux_lots[0],
              'ordre': r.oid[0], 'occurrence': r.occ[0], 'action préc.': r.prev_action[0]}).head(20).round(2)

In [ ]:
# Combien d'événements vit un ordre dans une fenêtre, et comment apparaît-il la première fois ?
O, on = lab.table(['orders'])
df = pd.DataFrame(O, columns=on)
fig, axs = plt.subplots(1, 2, figsize=(11, 3.2))
axs[0].hist(df['orders:orders_per_event'] * 100, bins=40, color=viz.PAL['blue']); axs[0].set_title('Ordres distincts par fenêtre'); axs[0].set_xlabel('nombre d\'ordres')
df[['orders:first_seen_A', 'orders:first_seen_U', 'orders:first_seen_D']].mean().rename(lambda s: s.split('_')[-1]).plot.bar(ax=axs[1], color=[viz.PAL['new'], viz.PAL['purple'], viz.PAL['old']])
axs[1].set_title('Première apparition d\'un ordre (U ou D = ordre antérieur à la fenêtre)'); axs[1].set_ylabel('part des ordres')
viz.save(fig, 'd1_orders')

## Partie 2 · Signatures — Les signatures des titres

**Question.** Qu'est-ce qui distingue un titre d'un autre, et est-ce que ça tient entre le train et le test ?

Le test n'a pas de labels. Pour le découper par titre, on utilise le titre **prédit** par la meilleure soumission (V3 B, 0,635). C'est approximatif, mais suffisant pour voir les tendances.

Durée : < 3 min.

In [ ]:
teacher = REPO / 'teacher' / 'v3_B_probs.npz'
if teacher.exists() and not DEMO:
    z = np.load(teacher); assert np.array_equal(z['obs_ids'], lab.test_ids)
    y_test_pred = z['p'].argmax(1); TEST_LABEL = 'test (titre prédit par V3 B, LB 0,635)'
else:
    y_test_pred, TEST_LABEL = None, None
print('titres du test :', TEST_LABEL or 'indisponibles (barres test masquées)')
def by_class(X, names, cols, y):
    df = pd.DataFrame(X, columns=names)[cols]; df['y'] = y; return df.groupby('y').mean()

In [ ]:
def shares(blocks, cols, title, name):
    Xtr, nm = lab.table(blocks); tr = by_class(Xtr, nm, cols, lab.y)
    te = by_class(lab.table(blocks, 'test')[0], nm, cols, y_test_pred) if y_test_pred is not None else tr * np.nan
    tr.columns = te.columns = [c.split(':')[1] for c in cols]
    return viz.per_class_shares(tr, te, list(tr.columns), LABELS, title, name, TEST_LABEL or 'test')

shares(['ticks'], ['ticks:spread_1', 'ticks:spread_2', 'ticks:spread_3_4', 'ticks:spread_5p'],
       'Régime de tick : part du temps à 1, 2, 3–4, ≥5 ticks de spread', 'd2_ticks')
shares(['lots'], ['lots:flux_round', 'lots:flux_odd', 'lots:flux_mixed'], 'Conventions de taille : lots ronds, odd lots, tailles mixtes', 'd2_lots')
shares(['cat_freq'], [f'cat_freq:venue={v}' for v in range(1, 7)], 'Répartition des événements entre venues', 'd2_venues')
shares(['position'], ['position:pos_best', 'position:pos_d1', 'position:pos_d2', 'position:pos_d3_5', 'position:pos_d6p'],
       'Où arrivent les événements (distance au meilleur prix)', 'd2_position')

In [ ]:
# Profondeur par titre : train contre test
Xtr, nm = lab.table(['levels']); j = nm.index('levels:log_depth_q50')
fig, ax = plt.subplots(figsize=(11, 3.4))
pos = np.arange(len(LABELS))
ax.boxplot([Xtr[lab.y == c, j] for c in range(len(LABELS))], positions=pos - .2, widths=.35, showfliers=False,
           patch_artist=True, boxprops=dict(facecolor=viz.PAL['blue'], alpha=.6))
if y_test_pred is not None:
    Xte = lab.table(['levels'], 'test')[0]
    ax.boxplot([Xte[y_test_pred == c, j] for c in range(len(LABELS))], positions=pos + .2, widths=.35, showfliers=False,
               patch_artist=True, boxprops=dict(facecolor=viz.PAL['orange'], alpha=.6))
ax.set_xticks(pos, LABELS); ax.set_ylabel('log(1 + profondeur médiane / lot)')
ax.set_title('Profondeur affichée par titre — bleu : train, orange : ' + (TEST_LABEL or 'test indisponible'))
viz.save(fig, 'd2_depth')

In [ ]:
# Empreinte « sac de mots » : fréquence des tokens action × côté × zone, titres regroupés par ressemblance
from scipy.cluster.hierarchy import leaves_list, linkage
Xb, nb_ = lab.table(['tokens_bow'])
M = by_class(Xb, nb_, nb_, lab.y); M = (M - M.mean()) / M.std()
o = leaves_list(linkage(M.values, 'average'))
fig, ax = plt.subplots(figsize=(10, 6))
im = ax.imshow(M.values[o], aspect='auto', cmap='RdBu_r', vmin=-2, vmax=2)
ax.set_yticks(range(len(o)), np.asarray(LABELS)[o]); ax.set_xticks(range(M.shape[1]), [c.split(':')[1] for c in M.columns], rotation=90, fontsize=6)
ax.set_title('Empreinte de chaque titre (écarts-types par rapport à la moyenne des titres)'); ax.grid(False)
fig.colorbar(im, ax=ax, shrink=.7); viz.save(fig, 'd2_token_fingerprint')

## Partie 3 · Dérive — Ce qui change entre le train et le test

**Question.** Le test vient d'une autre période : qu'est-ce qui a bougé, et est-ce compatible avec l'hypothèse de prix plus élevés (moins d'actions affichées, plus d'odd lots) ?

Durée : environ 5 min.

In [ ]:
ALL = ['base_relative', 'cat_freq', 'levels', 'ticks', 'lots', 'position', 'orders', 'book_dyn']
uni = screen.univariate(lab.dir, lab.cfg, ALL, base=core.OLD_STATS)
cplots.screen_map(uni, lab.dir); viz.save(plt.gcf(), 'd3_utility_vs_drift')
uni.sort_values('ks', ascending=False).head(12)[['feature', 'eta2', 'ks', 'shift_sd']].round(3)

In [ ]:
top = uni.sort_values('ks', ascending=False).head(6).feature.tolist()
fig, axs = plt.subplots(2, 3, figsize=(11, 5.4))
for a, f in zip(axs.ravel(), top):
    blk, col = f.split(':'); Xtr, nm = lab.table([blk]); Xte, _ = lab.table([blk], 'test'); j = nm.index(f)
    lo, hi = np.nanquantile(np.r_[Xtr[:, j], Xte[:, j]], [.005, .995]); bins = np.linspace(lo, hi, 40)
    a.hist(Xtr[:, j], bins, density=True, histtype='step', color=viz.PAL['blue'], label='train')
    a.hist(Xte[:, j], bins, density=True, histtype='step', color=viz.PAL['orange'], label='test'); a.set_title(f, fontsize=8)
axs[0, 0].legend(fontsize=7); fig.suptitle('Les 6 colonnes les plus décalées (KS)', fontweight='bold'); viz.save(fig, 'd3_top_shifts')

In [ ]:
# Hypothèse « prix plus élevés » : moins de profondeur ET plus d'odd lots doivent aller ensemble
L, ln = lab.table(['levels', 'lots']); Lt, _ = lab.table(['levels', 'lots'], 'test')
d, o = ln.index('levels:log_depth_q50'), ln.index('lots:flux_odd')
fig, axs = plt.subplots(1, 2, figsize=(11, 3.8), sharex=True, sharey=True)
for a, M_, t in [(axs[0], L, 'train'), (axs[1], Lt, 'test')]:
    a.hexbin(M_[:, d], M_[:, o], gridsize=40, cmap='Blues', mincnt=1, bins='log'); a.set_title(t)
    a.set_xlabel('log(1 + profondeur médiane / lot)')
axs[0].set_ylabel('part d\'odd lots dans le flux')
fig.suptitle('Profondeur contre odd lots', fontweight='bold'); viz.save(fig, 'd3_depth_vs_oddlots')
print('médianes — profondeur : train %.3f / test %.3f | odd lots : train %.3f / test %.3f' %
      (np.nanmedian(L[:, d]), np.nanmedian(Lt[:, d]), np.nanmean(L[:, o]), np.nanmean(Lt[:, o])))

In [ ]:
adv = screen.adversarial(lab.dir, lab.cfg, ALL, n=20000, device='cuda' if core.has_cuda() else 'cpu')
cplots.adversarial_bars(adv, lab.dir); viz.save(plt.gcf(), 'd3_adversarial'); adv.round(3)

## Partie 4 · Ordres — Comment vivent les ordres

**Question.** Les titres se distinguent-ils par la façon dont leurs ordres sont ajoutés, modifiés, annulés ou exécutés ?

Tout est mesuré **dans** des fenêtres tronquées de 100 événements : ce ne sont pas des durées de vie complètes.

Durée : < 2 min.

In [ ]:
O, on = lab.table(['orders'])
trans = [c for c in on if '_to_' in c]
M = by_class(O, on, trans, lab.y)
fig, axs = plt.subplots(4, 6, figsize=(12, 8))
for c, a in zip(range(len(LABELS)), axs.ravel()):
    a.imshow(M.loc[c].values.reshape(3, 3), cmap='Purples', vmin=0, vmax=M.values.max())
    a.set_xticks(range(3), list('ADU'), fontsize=6); a.set_yticks(range(3), list('ADU'), fontsize=6); a.set_title(f'titre {LABELS[c]}', fontsize=7); a.grid(False)
fig.suptitle('Transitions d\'un même ordre (ligne : action précédente → colonne : action suivante)', fontweight='bold')
viz.save(fig, 'd4_transitions_by_stock')

In [ ]:
cols = ['orders:repeat_share', 'orders:multi_event_orders', 'orders:added_then_deleted', 'orders:D_trade_share', 'orders:cross_venue_repeat']
tr = by_class(O, on, cols, lab.y)
fig, ax = plt.subplots(figsize=(11, 3.4))
for j, c in enumerate(cols):
    ax.plot(range(len(LABELS)), tr[c].values, 'o-', label=c.split(':')[1], color=viz.CYCLE[j])
ax.set_xticks(range(len(LABELS)), LABELS); ax.set_xlabel('Titre'); ax.set_ylabel('part'); ax.legend(fontsize=7, ncol=3)
ax.set_title('Profil des parcours d\'ordres par titre (train)'); viz.save(fig, 'd4_profiles')

In [ ]:
if y_test_pred is not None:
    te = by_class(lab.table(['orders'], 'test')[0], on, cols, y_test_pred)
    fig, axs = plt.subplots(1, len(cols), figsize=(13, 2.8))
    for a, c in zip(axs, cols):
        a.scatter(tr[c], te[c], s=14, color=viz.PAL['purple']); lim = [min(tr[c].min(), te[c].min()), max(tr[c].max(), te[c].max())]
        a.plot(lim, lim, 'k:', lw=.8); a.set_title(c.split(':')[1], fontsize=7); a.set_xlabel('train'); a.set_ylabel('test')
    fig.suptitle('Stabilité des parcours d\'ordres : chaque point est un titre (sur la diagonale = stable)', fontweight='bold')
    viz.save(fig, 'd4_train_vs_test')

In [ ]:
import shutil
archive = shutil.make_archive(str(viz.FIG_DIR), 'zip', viz.FIG_DIR)
print('Figures :', sorted(p.name for p in viz.FIG_DIR.glob('*.png')), '→', archive)